<a href="https://colab.research.google.com/github/sdfbk/90-Days-Big-Data-Analysis-Challenge/blob/main/Taxi_Trip_Record_Dataset_(Pyspark).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [63]:
# 1. Ingest with PySpark

In [87]:
spark = SparkSession.builder\
        .appName("Taxi Trip Mini Script")\
        .getOrCreate()

In [65]:
df_taxi = spark.read.parquet("/content/drive/MyDrive/yellow_tripdata_2026-01.parquet")
df_taxi.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [66]:
# 2.Clean and Aggregate

In [67]:
df_taxi.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [68]:
df_taxi.count()

3724889

In [69]:
# Calculate null counts for each column
null_counts = df_taxi.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_taxi.columns])

# Display the null counts
null_counts.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        1088058|            0|   1088058|           1088058|           0|    

In [70]:
# Display no of rows and columns
no_of_rows = df_taxi.count()
no_of_columns = len(df_taxi.columns)
print("Number of rows and columns in the Dataset:", no_of_rows,"and",no_of_columns)

Number of rows and columns in the Dataset: 3724889 and 20


In [71]:
# Display Discription of dataset
default = df_taxi.summary()
default.show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------

In [72]:
# Perform imputation
df_taxi = df_taxi.fillna({"passenger_count": 0, "RatecodeID":1, "store_and_fwd_flag":"N","congestion_surcharge":0.0,"Airport_fee":0.0})

In [73]:
check_nullcounts_again = df_taxi.select([F.sum(F.col(i).isNull().cast("int")).alias(i) for i in df_taxi.columns])
check_nullcounts_again.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|              0|            0|         0|                 0|           0|    

In [74]:
# Extract year and month and create new columns (feature engineering)
df_taxi = df_taxi.withColumn("trip_start_year",F.year("tpep_pickup_datetime"))\
                 .withColumn("trip_start_month",F.month("tpep_pickup_datetime"))

In [75]:
df_taxi.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+----------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_start_year|trip_start_month|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+----------------+
|       2| 2026-01-01 00:54:04|

In [76]:
default = df_taxi.summary()
default.show()

+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+-------------------+------------------+--------------------+--------------------+
|summary|          VendorID|   passenger_count|    trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|        Airport_fee|cbd_congestion_fee|     trip_start_year|    trip_start_month|
+-------+------------------+------------------+-----------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+---

In [77]:
#Aggregations and perform analysis
from pyspark.sql import functions as F

# 1. Extract date and hour features for granular grouping
df_taxi = df_taxi \
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime")) \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

In [78]:
# Group by pickup location, date, and hour, then compute metrics
total_trips = df_taxi.groupBy(
    "PULocationID",
    "pickup_date",
    "pickup_hour"
).agg(
    F.count("*").alias("total_trips")
)

total_trips.show(5)

+------------+-----------+-----------+-----------+
|PULocationID|pickup_date|pickup_hour|total_trips|
+------------+-----------+-----------+-----------+
|         133| 2026-01-01|          0|          4|
|         193| 2026-01-01|          3|          5|
|         226| 2026-01-01|          9|          8|
|          91| 2026-01-01|         10|          6|
|         239| 2026-01-01|         11|        129|
+------------+-----------+-----------+-----------+
only showing top 5 rows


In [79]:
total_revenue = df_taxi.groupBy("PULocationID","pickup_date","pickup_hour").agg(F.sum("total_amount").alias("total_revenue"))
total_revenue.show(5)

+------------+-----------+-----------+-------------+
|PULocationID|pickup_date|pickup_hour|total_revenue|
+------------+-----------+-----------+-------------+
|         133| 2026-01-01|          0|       195.45|
|         193| 2026-01-01|          3|       136.06|
|         226| 2026-01-01|          9|       263.01|
|          91| 2026-01-01|         10|       182.69|
|         239| 2026-01-01|         11|      2778.88|
+------------+-----------+-----------+-------------+
only showing top 5 rows


In [80]:
avg_fare_amount = df_taxi.groupBy(
    "PULocationID",
    "pickup_date",
    "pickup_hour"
).agg(
    F.avg("fare_amount").alias("avg_fare_amount")
)

avg_fare_amount.show(5)

+------------+-----------+-----------+------------------+
|PULocationID|pickup_date|pickup_hour|   avg_fare_amount|
+------------+-----------+-----------+------------------+
|         133| 2026-01-01|          0|           41.2775|
|         193| 2026-01-01|          3|            23.924|
|         226| 2026-01-01|          9|25.541249999999998|
|          91| 2026-01-01|         10|25.816666666666666|
|         239| 2026-01-01|         11|14.694728682170542|
+------------+-----------+-----------+------------------+
only showing top 5 rows


In [81]:
avg_trip_distance = df_taxi.groupBy(
    "PULocationID",
    "pickup_date",
    "pickup_hour"
).agg(
    F.avg("trip_distance").alias("avg_trip_distance")
)

avg_trip_distance.show(5)

+------------+-----------+-----------+------------------+
|PULocationID|pickup_date|pickup_hour| avg_trip_distance|
+------------+-----------+-----------+------------------+
|         133| 2026-01-01|          0|            7.0325|
|         193| 2026-01-01|          3|             4.128|
|         226| 2026-01-01|          9|3.1925000000000003|
|          91| 2026-01-01|         10|              5.59|
|         239| 2026-01-01|         11|2.4968992248062016|
+------------+-----------+-----------+------------------+
only showing top 5 rows


In [82]:
# Join all aggregated metrics into a single DataFrame
df_agg = total_trips \
    .join(total_revenue, on=["PULocationID", "pickup_date", "pickup_hour"], how="inner") \
    .join(avg_fare_amount, on=["PULocationID", "pickup_date", "pickup_hour"], how="inner") \
    .join(avg_trip_distance, on=["PULocationID", "pickup_date", "pickup_hour"], how="inner")

df_agg.show(5)

+------------+-----------+-----------+-----------+-------------+------------------+------------------+
|PULocationID|pickup_date|pickup_hour|total_trips|total_revenue|   avg_fare_amount| avg_trip_distance|
+------------+-----------+-----------+-----------+-------------+------------------+------------------+
|         133| 2026-01-01|          0|          4|       195.45|           41.2775|            7.0325|
|         193| 2026-01-01|          3|          5|       136.06|            23.924|             4.128|
|         226| 2026-01-01|          9|          8|       263.01|25.541249999999998|3.1925000000000003|
|          91| 2026-01-01|         10|          6|       182.69|25.816666666666666|              5.59|
|         239| 2026-01-01|         11|        129|      2778.88|14.694728682170542|2.4968992248062016|
+------------+-----------+-----------+-----------+-------------+------------------+------------------+
only showing top 5 rows


In [83]:
# join df_agg to main dataset to get whole data
df_taxi = df_agg.join(df_taxi,on=["PULocationID","pickup_date","pickup_hour"],how="inner")
df_taxi.show(5)

+------------+-----------+-----------+-----------+-------------+---------------+-----------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------+----------------+
|PULocationID|pickup_date|pickup_hour|total_trips|total_revenue|avg_fare_amount|avg_trip_distance|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_start_year|trip_start_month|
+------------+-----------+-----------+-----------+-------------+---------------+-----------------+--------+--------------------+---------------------+---------------+-------------+------

In [84]:
df_taxi.printSchema()

root
 |-- PULocationID: integer (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- total_trips: long (nullable = false)
 |-- total_revenue: double (nullable = true)
 |-- avg_fare_amount: double (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = false)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = 

In [85]:
# 3.Save to Parquet Partitioned by Year/Month
df_taxi.write.mode("overwrite").partitionBy("trip_start_year","trip_start_month").parquet("/content/drive/MyDrive/yellow_tripdata_2026-01_agg.parquet")

In [91]:
df_taxi_partitioned_2025 = spark.read.parquet("/content/drive/MyDrive/yellow_tripdata_2026-01_agg.parquet/trip_start_year=2025")
df_taxi_partitioned_2025.show(5)

+------------+-----------+-----------+-----------+-------------+-----------------+------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+
|PULocationID|pickup_date|pickup_hour|total_trips|total_revenue|  avg_fare_amount| avg_trip_distance|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_start_month|
+------------+-----------+-----------+-----------+-------------+-----------------+------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------

In [93]:
df_taxi_partitioned_2026 = spark.read.parquet("/content/drive/MyDrive/yellow_tripdata_2026-01_agg.parquet/trip_start_year=2026")
df_taxi_partitioned_2026.show()

+------------+-----------+-----------+-----------+-------------+------------------+------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------------+
|PULocationID|pickup_date|pickup_hour|total_trips|total_revenue|   avg_fare_amount| avg_trip_distance|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|trip_start_month|
+------------+-----------+-----------+-----------+-------------+------------------+------------------+--------+--------------------+---------------------+---------------+-------------+----------+---------------